In [11]:
import re

# --- Data Structures ---
class Term:
    pass

class Variable(Term):
    def __init__(self, name):
        self.name = name
    def __str__(self):
        return self.name
    def __repr__(self):
        return self.name
    def __eq__(self, other):
        return isinstance(other, Variable) and self.name == other.name
    def __hash__(self):
        return hash(self.name)

class Constant(Term):
    def __init__(self, name):
        self.name = name
    def __str__(self):
        return self.name
    def __repr__(self):
        return self.name
    def __eq__(self, other):
        return isinstance(other, Constant) and self.name == other.name
    def __hash__(self):
        return hash(self.name)

class Function(Term):
    def __init__(self, name, args):
        self.name = name
        self.args = list(args)
    def __str__(self):
        return f"{self.name}({', '.join(str(arg) for arg in self.args)})"
    def __repr__(self):
        return str(self)
    def __eq__(self, other):
        return isinstance(other, Function) and self.name == other.name and self.args == other.args

# --- Parser for expressions ---
def parse_term(tokens):
    if not tokens:
        raise ValueError("Unexpected end of input")
    token = tokens.pop(0)
    if not tokens or tokens[0] != '(':
        if token.islower() and token in {'u', 'v', 'w', 'x', 'y', 'z'}:
            return Variable(token)
        else:
            return Constant(token)
    else:
        name = token
        tokens.pop(0)  # consume '('
        args = []
        while True:
            args.append(parse_term(tokens))
            next_token = tokens.pop(0)
            if next_token == ')':
                break
            elif next_token != ',':
                raise ValueError(f"Expected ',' or ')' but got '{next_token}'")
        return Function(name, args)

def parse_expression(expr_str):
    expr_str = expr_str.replace(' ', '')
    tokens = re.findall(r'[a-zA-Z0-9]+|[,()]', expr_str)
    return parse_term(tokens)

# --- Unification Logic ---
def occurs_check(var, term, subst):
    if var == term:
        return True
    elif isinstance(term, Variable):
        if term in subst:
            return occurs_check(var, subst[term], subst)
    elif isinstance(term, Function):
        return any(occurs_check(var, arg, subst) for arg in term.args)
    return False

def apply_substitution(term, subst):
    if isinstance(term, Variable):
        if term in subst:
            return apply_substitution(subst[term], subst)
        return term
    elif isinstance(term, Function):
        return Function(term.name, [apply_substitution(arg, subst) for arg in term.args])
    return term

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if subst is False:
        return False
    if x == y:
        return subst
    if isinstance(x, Variable):
        return unify_var(x, y, subst)
    if isinstance(y, Variable):
        return unify_var(y, x, subst)
    if isinstance(x, Function) and isinstance(y, Function):
        if x.name != y.name or len(x.args) != len(y.args):
            return False
        temp_subst = dict(subst)
        for arg_x, arg_y in zip(x.args, y.args):
            temp_subst = unify(arg_x, arg_y, temp_subst)
            if temp_subst is False:
                return False
        return temp_subst
    return False

def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    if isinstance(val, Variable) and val in subst:
        return unify(var, subst[val], subst)
    if occurs_check(var, val, subst):
        return False
    new_subst = dict(subst)
    new_subst[var] = val
    return new_subst

def resolve_substitutions(subst):
    resolved = {}
    for var in subst:
        resolved[var] = apply_substitution(var, subst)
    return resolved

# --- Interactive Input ---
print("=== FOL Unification Solver ===")
print("Convention: u, v, w, x, y, z are treated as variables; other terms/symbols as constants/functions.")
expression_1 = input("Enter first expression (e.g., f(x, g(y))): ")
expression_2 = input("Enter second expression (e.g., f(h(z), g(a))): ")

try:
    t1 = parse_expression(expression_1)
    t2 = parse_expression(expression_2)
    print(f"\nParsed Term 1: {t1}")
    print(f"Parsed Term 2: {t2}")

    mgu = unify(t1, t2)
    if mgu is False:
        print("\nResult: Unification Failed!")
    else:
        print("\nResult: Unification Succeeded!")
        resolved_mgu = resolve_substitutions(mgu)
        print("Most General Unifier (MGU):")
        for var, val in resolved_mgu.items():
            print(f"  {var} -> {val}")

        # Apply MGU to the original expression to show the final clean unified expression
        unified_expr = apply_substitution(t1, mgu)
        print(f"\nUnified Expression: {unified_expr}")
except Exception as e:
    print(f"\nError parsing expressions: {e}")



=== FOL Unification Solver ===
Convention: u, v, w, x, y, z are treated as variables; other terms/symbols as constants/functions.
Enter first expression (e.g., f(x, g(y))):  f(x, g(y)))
Enter second expression (e.g., f(h(z), g(a))): Q(Y,G,(A,B,B))

Parsed Term 1: f(x, g(y))
Parsed Term 2: Q(Y, G, group(A, B, B))

Result: Unification Failed!
